In [ ]:
import random
import json
import re
import os
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as tdist
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.common_utils import extract_score
from local_variables import TIME_SERIES_CATEGORIES, TREND_DIRECTIONS



def trend_test_ols(y, maxlags=None, alpha=0.05):
    """
    Test for a linear trend in y using OLS with HAC (Newey–West) SEs.

    H0: slope == 0
    H1: slope != 0  (two-sided)

    Returns
    -------
    slope : float
        Estimated slope of the linear trend.
    p_two_sided : float
        Two-sided p-value for H0: slope == 0.
    direction : {'increasing','decreasing','neither'}
        Direction of trend if statistically significant at alpha; else 'neither'.
    significant : bool
        True if p_two_sided < alpha, else False.
    """
    y = np.asarray(y, dtype=float)
    x = np.arange(len(y))
    X = sm.add_constant(x)
    model = sm.OLS(y, X, missing='drop')

    # Choose HAC lag ~ T^(1/4) if not supplied
    T = np.isfinite(y).sum()
    if maxlags is None:
        maxlags = max(1, int(T ** 0.25))

    res = model.fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    slope = float(res.params[1])
    tval = float(res.tvalues[1])
    df = float(res.df_resid)

    # One-sided p-values for convenience
    p_up = 1 - tdist.cdf(tval, df=df)   # H1: slope > 0
    p_down = tdist.cdf(tval, df=df)     # H1: slope < 0

    # Two-sided p-value
    p_two_sided = 2 * min(p_up, p_down)

    significant = p_two_sided < alpha
    if significant:
        direction = 'increasing' if tval > 0 else 'decreasing'
    else:
        direction = 'neither'

    return slope, p_two_sided, direction, significant

In [ ]:
def generate_trend(start, end, noise_std=1, time_series_length=52):
    """
    Generate synthetic trend data.
    :param start: Starting value of the trend.
    :param end: Ending value of the trend.
    :param noise_std: std of random noise to add.
    :param time_series_length: Number of data points to generate.
    :return: List of trend values.
    """
    trend = np.linspace(start, end, time_series_length)  
    noise = np.random.normal(0, noise_std, time_series_length)
    return trend + noise
# Example usage
trend_data = generate_trend(100, 120, 15, time_series_length=52)
plt.plot(trend_data)
plt.show()    
np.std(trend_data)
# Example:
slope, p, direction, sig = trend_test_ols(trend_data)
print(f"slope={slope:.4f}, p={p:.4g}, direction={direction}, significant? {sig}")


In [ ]:
n = 10
offset = 20
noised_std = 15
starting_level = 100
output_dir = "data"
time_series_length = 52
for time_series_category in TIME_SERIES_CATEGORIES:
    for trend in TREND_DIRECTIONS:
        for i in range(1,n+1):
            if trend == "up":
                data = generate_trend(starting_level, starting_level+offset, noise_std=noised_std, time_series_length=52).round(2)
            elif trend == "down":
                data = generate_trend(starting_level, starting_level-offset, noise_std=noised_std, time_series_length=52).round(2)
            else:  # stable
                data = generate_trend(starting_level, starting_level, noise_std=noised_std, time_series_length=52).round(2)
            plt.plot(data, label=f"{time_series_category} - {trend}")
            plt.title(f"{time_series_category}")
            # plot weeks 1 to 52 on x-axis every 4 weeks
            plt.xticks(ticks=np.arange(0,len(data)+1,4), labels=[f"{i}" for i in range(0, len(data)+1, 4)])
            plt.xlabel("Weeks")
            plt.savefig(f"{output_dir}/{time_series_category}_{trend}_{i}.png")

            slope, p_two_sided, direction, significant = trend_test_ols(data, maxlags=None, alpha=0.05)
            text = f"{trend}: {time_series_category}: slope={slope:.2f}, p-value={p_two_sided:.2f},\ndirection={direction}, significant={significant}"
            plt.figtext(0.14, 0.8, text, wrap=True, horizontalalignment='left', fontsize=10)
            # save plot to file
            if significant:
                # save data to csv and index from Week1 to Week52
                df = pd.DataFrame({"value": data}, index=[f"Week{i+1}" for i in range(len(data))])
                df.to_csv(f"{output_dir}/{time_series_category}_{trend}_{i}.csv")

            plt.show()